# Airbnb Paris – Experiment 1: PyOD-Baselines
- iForest, LODA, ECOD, AutoEncoder (unsupervised, Originalverteilung, ohne Labels)
- Grid Search auf Val nach AUPRC; gefittet wird ausschließlich auf Train, Test wird genau einmal ausgewertet

In [ ]:
import os
import time
import numpy as np
import pandas as pd
import mlflow
from sklearn.model_selection import ParameterGrid
from sklearn.metrics import roc_auc_score, classification_report, precision_recall_curve, auc
from pyod.models.iforest import IForest
from pyod.models.loda import LODA
from pyod.models.ecod import ECOD
from pyod.models.auto_encoder import AutoEncoder

SEED = int(os.environ.get("SEED", 1))
print("SEED", SEED)

## Daten & Split laden
- Outlier = 1; die Split-Zuordnung kommt aus der Split-Datei, es wird nicht neu gesplittet

In [ ]:
df = pd.read_csv(f"../../data/preprocessed/cleaned_airbnb_paris_seed{SEED}.csv")
split = pd.read_csv(f"../../data/splits/split_airbnb_paris_seed{SEED}.csv")
y = (1 - df["is_top_rating"]).values
X = df.drop(columns=["row_id", "is_top_rating"]).values

s = df["row_id"].map(split.set_index("row_id")["split"]).values
X_tr, y_tr = X[s == "train"], y[s == "train"]
X_va, y_va = X[s == "val"], y[s == "val"]
X_te, y_te = X[s == "test"], y[s == "test"]
print("train", X_tr.shape, "val", X_va.shape, "test", X_te.shape, "| Outlier-Rate Test:", round(y_te.mean(), 4))

## MLflow

In [ ]:
mlflow.set_tracking_uri("file:../../mlruns")
mlflow.set_experiment("airbnb_paris_experiment_1")

## Grid Search auf Val (AUPRC), Refit auf Train, Test auswerten
- Schwelle für den Classification Report: Top-k Scores mit k = Testgröße × Outlier-Rate im Train (nutzt nur Train-Information)

In [ ]:
models = {
    "iforest": (IForest, {"n_estimators": [100, 200], "max_features": [0.5, 1.0], "random_state": [SEED]}),
    "loda": (LODA, {"n_bins": [10, 20], "n_random_cuts": [100, 200]}),
    "ecod": (ECOD, {}),
    "autoencoder": (AutoEncoder, {"hidden_neuron_list": [[64, 32], [32, 16]], "epoch_num": [20, 50],
                                  "random_state": [SEED]}),
}
k = round(len(y_te) * y_tr.mean())

for name, (Model, grid) in models.items():
    t0 = time.perf_counter()
    best_auprc, best_params = -1.0, {}
    for params in (list(ParameterGrid(grid)) or [{}]):
        m = Model(**params)
        m.fit(X_tr)
        p, r, _ = precision_recall_curve(y_va, m.decision_function(X_va))
        val_auprc = auc(r, p)
        if val_auprc > best_auprc:
            best_auprc, best_params = val_auprc, params

    model = Model(**best_params)
    model.fit(X_tr)
    scores = model.decision_function(X_te)
    runtime = time.perf_counter() - t0

    prec, rec, _ = precision_recall_curve(y_te, scores)
    auprc = auc(rec, prec)
    auroc = roc_auc_score(y_te, scores)
    pred = (scores >= np.sort(scores)[-k]).astype(int)

    with mlflow.start_run(run_name=name):
        mlflow.log_params({**best_params, "seed": SEED})
        mlflow.log_metric("auprc", auprc)
        mlflow.log_metric("auc_roc", auroc)
        mlflow.log_metric("runtime_s", runtime)
    print(f"{name}: AUPRC={auprc:.4f} AUC-ROC={auroc:.4f} time={runtime:.1f}s params={best_params}")
    print(classification_report(y_te, pred, target_names=["inlier", "outlier"], digits=4, zero_division=0))